# SCOPUS Collection Pipeline

*Author: Regina Chua*

> This notebook is the SCOPUS arm of the systematic review. It mirrors `pubmed.ipynb` so the two
> pipelines stay comparable: the same `(disease) AND (spatial) AND (exposure) NOT (exclusions)`
> logic is reused, only translated into SCOPUS' Advanced Search syntax. Everything that defines
> *what* we search for is imported from `search_strategy.py` so the query stays in sync with every
> other database.

SCOPUS is accessed here through [**elsapy**](https://github.com/ElsevierDev/elsapy), Elsevier's own
lightweight Python wrapper around `api.elsevier.com`. It defaults to 25 results per page — the
free-tier limit — and follows pagination links automatically, avoiding the `Scopus400Error` that
occurs when a subscriber-mode client requests a larger page size without an InstToken.

> **Note:** elsapy was archived by Elsevier in January 2025. It is still fully functional but will
> not receive further updates. The Elsevier API it talks to has not changed.

**Access note:** abstract text and author keywords are only available under the `COMPLETE` view,
which requires a subscriber InstToken (on-campus or VPN). `STANDARD` view — used by default — still
returns titles, DOIs, journals, and citation counts.

**References:** [elsapy on GitHub](https://github.com/ElsevierDev/elsapy),
[Scopus Search API](https://dev.elsevier.com/documentation/ScopusSearchAPI.wadl).

## 1. Environment Setup

> `ElsClient` takes the API key and optional InstToken directly. I read the key from the existing
> pybliometrics config (`~/.config/pybliometrics.cfg`) as a convenience — you've already stored it
> there — with a fallback to `ELSEVIER_API_KEY` in `.env`. The masked key is printed so it's easy
> to confirm the right one loaded. Without an InstToken the API caps at 5,000 results per query;
> Section 3 handles this by collecting one year at a time.

In [1]:
import os
import re
from configparser import ConfigParser
from datetime import datetime
from pathlib import Path

import pandas as pd
from dotenv import load_dotenv
from elsapy.elsclient import ElsClient
from elsapy.elssearch import ElsSearch

In [2]:
from search_strategy import (
    INCLUSION_CRITERIA,
    ALTERNATE_TERMS,
    EXCLUSION_TERMS,
    DATE_FILTER,
    CLEANING_RULES,
)

load_dotenv()
pd.set_option("display.max_colwidth", 120)

In [3]:
# --- Load API key: pybliometrics config first, .env as override ---
def _read_pyb_config(path="~/.config/pybliometrics.cfg"):
    cfg = ConfigParser()
    cfg.read(Path(path).expanduser())
    return (
        cfg.get("Authentication", "APIKey",    fallback=None),
        cfg.get("Authentication", "InstToken", fallback=None),
    )

pyb_key, pyb_token = _read_pyb_config()
API_KEY   = os.getenv("ELSEVIER_API_KEY")   or pyb_key
INST_TOKEN = os.getenv("ELSEVIER_INSTTOKEN") or pyb_token or None

In [6]:
SCOPUS_READY = bool(API_KEY)
if API_KEY:
    masked = API_KEY[:4] + "..." + API_KEY[-4:]
    print(f"API key loaded: {bool(API_KEY)}")
    print(f"InstToken set:  {bool(INST_TOKEN)}")
    if not INST_TOKEN:
        print(
            "\n  Without an InstToken the API is limited to 5,000 results per query.\n"
            "  Collection will run year-by-year to stay within this limit.\n"
            "  For subscriber access (uncapped + COMPLETE view), ask your library for\n"
            "  an InstToken and add it to .env as ELSEVIER_INSTTOKEN=<token>."
        )
else:
    print(
        "No API key found. Add ELSEVIER_API_KEY=... to .env\n"
        "or run pybliometrics.init() to configure the pybliometrics config file."
    )

print("\nEnvironment ready.")

API key loaded: True
InstToken set:  True

Environment ready.


## 2. Build Query

> Same translation as before: PubMed's `[TIAB]` → SCOPUS' `TITLE-ABS-KEY(...)`, with `PUBYEAR`
> bounds for the date window. The year-by-year splitting in Section 3 replaces the range bounds with
> a single-year constraint per query, so this base query is built *without* `PUBYEAR` appended —
> the collection loop adds those dynamically.

In [8]:
def merge_terms(primary, alternates):
    """Combine primary + alternate/NLP terms per category (order-preserving,
    case-insensitive de-duplication). Mirrors pubmed.ipynb."""
    merged = {}
    for category, base_terms in primary.items():
        seen, combined = set(), []
        for t in list(base_terms) + list(alternates.get(category, [])):
            if t.lower() not in seen:
                seen.add(t.lower())
                combined.append(t)
        merged[category] = combined
    return merged


def tak_group(terms):
    """Wrap terms in a SCOPUS TITLE-ABS-KEY() OR-group."""
    return "TITLE-ABS-KEY(" + " OR ".join(f'\"{t}\"' for t in terms) + ")"


def title_group(terms):
    """Wrap terms in a SCOPUS TITLE() OR-group (title field only)."""
    return "TITLE(" + " OR ".join(f'\"{t}\"' for t in terms) + ")"


def build_base_query(inclusion, exclusion):
    """Compose the SCOPUS query without date bounds (added per-year in Section 3)."""
    include = " AND ".join([
        title_group(inclusion["disease"]),
        tak_group(inclusion["spatial"]),
        tak_group(inclusion["exposure"]),
    ])
    return f"{include} AND NOT {tak_group(exclusion)}"


INCLUDE_ALTERNATE_TERMS = False
search_criteria = (
    merge_terms(INCLUSION_CRITERIA, ALTERNATE_TERMS)
    if INCLUDE_ALTERNATE_TERMS
    else INCLUSION_CRITERIA
)

base_query = build_base_query(search_criteria, EXCLUSION_TERMS)

print("Alternate terms folded into query:", INCLUDE_ALTERNATE_TERMS)
for category, term_list in search_criteria.items():
    print(f"{category} terms ({len(term_list)}):", term_list)
print("\nBase query (PUBYEAR bounds added per year in Section 3):\n", base_query)

Alternate terms folded into query: False
disease terms (1): ['parkinson*']
spatial terms (25): ['environment*', 'geospatial*', 'geograph*', 'GIS', 'geographic information systems', 'spatiotemporal', 'spatial analysis', 'spatial interpolation', 'spatial epidemiology', 'remote sens*', 'latitude', 'longitude', 'clustering', 'residence', 'administrative division', 'drone', 'imagery', 'landsat', 'map', 'mapping', 'modis', 'satellite', 'sentinel', 'topolog*', 'altitude']
exposure terms (26): ['pollut*', 'chemical', 'pesticide*', 'air pollution', 'microplastic pollution', 'traffic pollution', 'water pollution', 'trichloroethylene', 'air quality', 'exposure', 'environment*', 'particulate*', 'atmospher*', 'carbon', 'humidity', 'meteorologic*', 'nitrogen*', 'ozone', 'PM2.5', 'PM10', 'surface pressure', 'temperature', 'heavy metals', 'ambient air pollution', 'long*term exposure', 'longitudinal']

Base query (PUBYEAR bounds added per year in Section 3):
 TITLE("parkinson*") AND TITLE-ABS-KEY("envi

## 3. Collect Articles from SCOPUS

> `ElsClient` defaults to 25 results per page — the free-tier page size — so it never hits the
> `Scopus400Error` that occurs when a subscriber-mode client requests a larger page size without
> an InstToken. `ElsSearch.execute(get_all=True)` then follows the `next` pagination links
> automatically until all results for that year are retrieved (up to the 5,000 cap).
>
> Running one query per year keeps each slice well under 5,000 so the full 2020–2025 window is
> covered. `elsapy` writes a `dump.json` sidecar after each search; I delete it after each year so
> it doesn't accumulate. Set `VIEW = "COMPLETE"` once you have an InstToken to also pull abstracts
> and author keywords.

In [10]:
import time
from requests.exceptions import HTTPError

VIEW = None  # None → STANDARD (titles/DOIs/journals). Set "COMPLETE" with InstToken for abstracts.


def flatten_scopus(rec):
    """Flatten one elsapy/Scopus result dict into the shared schema.

    elsapy returns the raw Scopus JSON, which uses Dublin Core (dc:) and
    PRISM (prism:) field prefixes.
    """
    doi = rec.get("prism:doi")
    return {
        "title":            rec.get("dc:title"),
        "abstract":         rec.get("dc:description"),   # COMPLETE view only
        "publication_date": rec.get("prism:coverDate"),
        "authors":          rec.get("dc:creator"),        # first-listed author only in STANDARD
        "journal":          rec.get("prism:publicationName"),
        "doi":              doi,
        "url":              f"https://doi.org/{doi}" if doi else None,
        "num_citations":    rec.get("citedby-count"),
        "pubmed_id":        rec.get("pubmed-id"),
        "keywords":         rec.get("authkeywords"),      # COMPLETE view only
        "source":           "scopus",
    }


run_ts = datetime.now().isoformat(timespec="seconds")
all_records = []

if not SCOPUS_READY:
    print("Skipping — API key not set (see Section 1).")
else:
    client = ElsClient(API_KEY, inst_token=INST_TOKEN)

    start_year = int(DATE_FILTER["start_date"][:4])
    end_year   = int(DATE_FILTER["end_date"][:4]) if DATE_FILTER.get("end_date") else datetime.now().year
    years = list(range(start_year, end_year + 1))

    print(f"Collecting year by year ({start_year}–{end_year}), view={'STANDARD' if not VIEW else VIEW}")
    print(f"Starting at {run_ts}\n")

    for yr in years:
        yr_query = f"{base_query} AND PUBYEAR > {yr - 1} AND PUBYEAR < {yr + 1}"
        search = ElsSearch(yr_query, "scopus")
        
        # Retry logic with exponential backoff (max 3 attempts)
        max_retries = 3
        retry_count = 0
        success = False
        
        while retry_count < max_retries and not success:
            try:
                search.execute(client, get_all=True, view=VIEW)
                results = search.results or []
                # Skip the sentinel 'no results' entry elsapy sometimes returns
                results = [r for r in results if r.get("dc:title") or r.get("prism:doi")]
                all_records.extend(flatten_scopus(r) for r in results)
                print(f"  {yr}: {search.tot_num_res} total → downloaded {len(results)}")
                success = True
            except HTTPError as exc:
                retry_count += 1
                if exc.response.status_code == 403:
                    wait_time = min(2 ** retry_count, 32)  # exponential backoff: 2, 4, 8, 16, 32 sec
                    print(f"  {yr}: HTTP 403 (Forbidden) — retrying in {wait_time}s (attempt {retry_count}/{max_retries})")
                    time.sleep(wait_time)
                else:
                    print(f"  {yr}: FAILED ({type(exc).__name__}: {exc})")
                    success = True  # don't retry for non-403 errors
            except Exception as exc:  # noqa: BLE001
                print(f"  {yr}: FAILED ({type(exc).__name__}: {exc})")
                success = True  # don't retry for other errors
            finally:
                # elsapy writes a dump.json sidecar after every execute() — clean it up
                dump = Path("dump.json")
                if dump.exists():
                    dump.unlink()
        
        # Rate limit: wait between years to avoid hitting quota
        if yr < years[-1]:
            time.sleep(2)

    df_raw = pd.DataFrame(all_records)
    print(f"\nTotal collected: {len(df_raw)} records across {len(years)} years.")

preview_cols = [c for c in ["title", "publication_date", "journal", "doi", "num_citations"]
                if c in df_raw.columns]
if not df_raw.empty:
    display(df_raw[preview_cols].head())

Starting at 2026-06-22T00:30:49



AttributeError: 'NoneType' object has no attribute 'status_code'

## 4. Clean Results

> Same cleaning contract as PubMed, driven by `CLEANING_RULES`: drop duplicate titles and (by
> default) require a DOI so every record is uniquely identifiable for cross-database deduplication
> later. SCOPUS does not always populate `pubmed_id`, so unlike PubMed I do **not** require it here.

In [ ]:
df_clean = df_raw.copy()

if not df_clean.empty:
    if CLEANING_RULES.get("remove_duplicate_titles", True):
        df_clean = df_clean.dropna(subset=["title"])
        df_clean["_title_lower"] = df_clean["title"].str.lower().str.strip()
        df_clean = df_clean.drop_duplicates(subset=["_title_lower"]).drop(columns=["_title_lower"])
    if CLEANING_RULES.get("require_doi", True) and "doi" in df_clean.columns:
        df_clean = df_clean.dropna(subset=["doi"])

print(f"Records after cleaning: {len(df_clean)}  (from {len(df_raw)} raw)")
if not df_clean.empty:
    display(df_clean[preview_cols].head())

## 5. Export

> Save the cleaned set to CSV — the stable hand-off artifact for the screening stage and for merging
> with PubMed/EMBASE/Web of Science in the deduplication step (Milestone 4). Only writes when there
> is data, so a failed run doesn't clobber a previous good export.

In [ ]:
output_path = Path("scopus_results_2026.csv")

if df_clean.empty:
    print("Nothing to export — df_clean is empty (see Sections 1 and 3).")
else:
    df_clean.to_csv(output_path, index=False)
    print(f"Exported {len(df_clean)} records to {output_path.resolve()}")
    print(f"Run timestamp: {run_ts}")

## 6. Notes & Next Steps

> - **Why elsapy instead of pybliometrics:** `pybliometrics` defaults to subscriber-mode pagination
>   (large page sizes) which triggers a `Scopus400Error` without an InstToken. elsapy uses 25
>   results per page — the free-tier limit — by default.
> - **Abstracts for screening:** `STANDARD` view does not return abstract text. Re-run with
>   `VIEW = "COMPLETE"` once you have an InstToken (`ELSEVIER_INSTTOKEN` in `.env`) — this is
>   needed before SCOPUS records can be LLM pre-screened in Milestone 2.
> - **InstToken:** ask your institution's library. It typically takes the form of a 32-character
>   hex string. Add it to `.env` as `ELSEVIER_INSTTOKEN=<token>` — Section 1 picks it up automatically.
> - **dump.json:** elsapy writes this sidecar after every `execute()`. Section 3 deletes it after
>   each year-slice so it does not accumulate.
> - **Deduplication:** export columns match the other database CSVs for clean concatenation in
>   Milestone 4.
> - **Query parity:** if `search_strategy.py` changes, re-run all collection notebooks together.